# COMP5241 · 学习数据速览

这个 notebook 用来把"每天花了多少时间学习"变成能看懂的结论：总时长、科目分布、最近 7 天趋势、专注连续天数。

**运行前提**

- 只需要 Python 标准库，`Kernel → Restart & Run All` 可以直接跑完。
- `pandas` / `matplotlib` 是可选的，装了会自动多画两张更标准的图，没装就跳过那一节。
- 数据默认用下面内置的示例；如果你把真实数据存成 `study_log.csv` 或 `study_data.json`（和本文件同一目录），notebook 会自动改读真实数据。

In [1]:
import csv
import json
import os
import platform
import statistics as stats
import sys
from collections import defaultdict
from datetime import date, timedelta

from IPython.display import HTML, display

print("Python", sys.version.split()[0], "·", platform.system())

optional = {}
for name in ("pandas", "numpy", "matplotlib"):
    try:
        module = __import__(name)
        optional[name] = getattr(module, "__version__", "installed")
    except ImportError:
        optional[name] = None

print("可选依赖：", {k: v for k, v in optional.items()})

Python 3.9.6 · Darwin
可选依赖： {'pandas': None, 'numpy': None, 'matplotlib': None}


## 1. 准备数据

每条记录四个字段：

| 字段 | 含义 | 示例 |
| --- | --- | --- |
| `day` | 日期，`YYYY-MM-DD` | `2026-09-17` |
| `subject` | 科目或类型 | `讲座` / `作业` / `复习` / `项目` / `实验` |
| `minutes` | 当天在该类型上投入的分钟数 | `95` |
| `tasks` | 当天完成的任务数 | `2` |

In [2]:
SAMPLE_LOG = [
    {"day": "2026-08-31", "subject": "讲座", "minutes": 60, "tasks": 0},
    {"day": "2026-09-01", "subject": "讲座", "minutes": 90, "tasks": 1},
    {"day": "2026-09-01", "subject": "复习", "minutes": 45, "tasks": 1},
    {"day": "2026-09-02", "subject": "作业", "minutes": 120, "tasks": 2},
    {"day": "2026-09-03", "subject": "讲座", "minutes": 60, "tasks": 0},
    {"day": "2026-09-04", "subject": "项目", "minutes": 150, "tasks": 1},
    {"day": "2026-09-05", "subject": "复习", "minutes": 40, "tasks": 1},
    {"day": "2026-09-06", "subject": "实验", "minutes": 110, "tasks": 2},
    {"day": "2026-09-07", "subject": "讲座", "minutes": 60, "tasks": 0},
    {"day": "2026-09-08", "subject": "作业", "minutes": 95, "tasks": 1},
    {"day": "2026-09-09", "subject": "复习", "minutes": 70, "tasks": 2},
    {"day": "2026-09-10", "subject": "项目", "minutes": 180, "tasks": 1},
    {"day": "2026-09-11", "subject": "讲座", "minutes": 60, "tasks": 0},
    {"day": "2026-09-12", "subject": "实验", "minutes": 130, "tasks": 2},
    {"day": "2026-09-13", "subject": "复习", "minutes": 85, "tasks": 1},
    {"day": "2026-09-14", "subject": "作业", "minutes": 100, "tasks": 2},
    {"day": "2026-09-15", "subject": "讲座", "minutes": 60, "tasks": 0},
    {"day": "2026-09-15", "subject": "项目", "minutes": 75, "tasks": 1},
    {"day": "2026-09-16", "subject": "复习", "minutes": 120, "tasks": 2},
    {"day": "2026-09-17", "subject": "作业", "minutes": 140, "tasks": 3},
    {"day": "2026-09-17", "subject": "实验", "minutes": 55, "tasks": 1},
]


def load_csv(path):
    records = []
    with open(path, newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            records.append(
                {
                    "day": row["day"].strip(),
                    "subject": row["subject"].strip(),
                    "minutes": int(row["minutes"]),
                    "tasks": int(row.get("tasks") or 0),
                }
            )
    return records


def load_web_export(path):
    """把看板导出的 localStorage JSON 转成记录。"""
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    records = []
    for day, minutes in sorted(payload.get("focusMinutes", {}).items()):
        if minutes:
            records.append({"day": day, "subject": "专注计时", "minutes": int(minutes), "tasks": 0})
    for day, count in sorted(payload.get("tasksDone", {}).items()):
        records.append({"day": day, "subject": "任务", "minutes": 0, "tasks": int(count)})
    return records


def load_records():
    if os.path.exists("study_log.csv"):
        return load_csv("study_log.csv"), "study_log.csv"
    if os.path.exists("study_data.json"):
        return load_web_export("study_data.json"), "study_data.json"
    return SAMPLE_LOG, "内置示例数据"


records, source = load_records()
print(f"数据来源：{source} · 共 {len(records)} 条记录")
records[:4]

数据来源：内置示例数据 · 共 21 条记录


[{'day': '2026-08-31', 'subject': '讲座', 'minutes': 60, 'tasks': 0},
 {'day': '2026-09-01', 'subject': '讲座', 'minutes': 90, 'tasks': 1},
 {'day': '2026-09-01', 'subject': '复习', 'minutes': 45, 'tasks': 1},
 {'day': '2026-09-02', 'subject': '作业', 'minutes': 120, 'tasks': 2}]

## 2. 基础统计

先把记录按天和按科目汇总，再看总数、日均、最长的一天和连续专注天数。

In [3]:
by_day = defaultdict(int)
by_subject = defaultdict(int)
tasks_by_subject = defaultdict(int)

for row in records:
    by_day[row["day"]] += row["minutes"]
    by_subject[row["subject"]] += row["minutes"]
    tasks_by_subject[row["subject"]] += row["tasks"]

days = sorted(by_day)
total_minutes = sum(by_day.values())
active_days = [day for day in days if by_day[day] > 0]
best_day = max(by_day, key=by_day.get)


def streak(day_keys):
    """从最近一天往回数，连续多少天有记录。"""
    if not day_keys:
        return 0
    keys = set(day_keys)
    cursor = date.fromisoformat(max(keys))
    count = 0
    while cursor.isoformat() in keys:
        count += 1
        cursor -= timedelta(days=1)
    return count


print(f"统计区间：{days[0]} → {days[-1]}（{len(days)} 天，其中有记录的 {len(active_days)} 天）")
print(f"总专注时长：{total_minutes} 分钟 ≈ {total_minutes / 60:.1f} 小时")
print(f"有记录日均：{total_minutes / len(active_days):.0f} 分钟")
print(f"最长的一天：{best_day} · {by_day[best_day]} 分钟")
print(f"连续专注：{streak(active_days)} 天")
print(f"完成任务：{sum(tasks_by_subject.values())} 个")

统计区间：2026-08-31 → 2026-09-17（18 天，其中有记录的 18 天）
总专注时长：1905 分钟 ≈ 31.8 小时
有记录日均：106 分钟
最长的一天：2026-09-17 · 195 分钟
连续专注：18 天
完成任务：24 个


In [4]:
def render_table(headers, rows, caption=""):
    """把二维数据渲染成一张简单的 HTML 表格。"""
    head = "".join(f"<th scope='col' style='text-align:left;padding:6px 12px;border-bottom:2px solid #99f6e4'>{h}</th>" for h in headers)
    body = []
    for row in rows:
        cells = "".join(
            f"<td style='padding:6px 12px;border-bottom:1px solid #e8f1f4'>{value}</td>" for value in row
        )
        body.append(f"<tr>{cells}</tr>")
    html = (
        "<table style='border-collapse:collapse;font-family:system-ui,sans-serif;font-size:14px'>"
        f"<caption style='text-align:left;padding-bottom:6px;font-weight:600'>{caption}</caption>"
        f"<thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table>"
    )
    display(HTML(html))


rows = []
for subject, minutes in sorted(by_subject.items(), key=lambda item: -item[1]):
    share = minutes / total_minutes * 100
    rows.append(
        [
            subject,
            f"{minutes} 分钟",
            f"{share:.1f}%",
            f"{tasks_by_subject[subject]} 个",
            "█" * round(share / 3),
        ]
    )

render_table(["科目", "时长", "占比", "完成任务", "占比图"], rows, caption="按科目汇总")

科目,时长,占比,完成任务,占比图
作业,455 分钟,23.9%,8 个,████████
项目,405 分钟,21.3%,3 个,███████
讲座,390 分钟,20.5%,1 个,███████
复习,360 分钟,18.9%,7 个,██████
实验,295 分钟,15.5%,5 个,█████


## 3. 可视化（不依赖 matplotlib）

下面两张图直接用内联 SVG 画出来，所以任何环境都能显示。

In [5]:
def bar_chart(labels, values, *, title="", color="#0d9488", highlight=None, highlight_color="#ea580c", unit=""):
    width, height = 780, 280
    pad_left, pad_top, pad_bottom = 52, 42, 42
    plot_w = width - pad_left - 20
    plot_h = height - pad_top - pad_bottom
    vmax = max(values) or 1
    slot = plot_w / len(values)
    bar_w = min(44.0, slot * 0.62)

    parts = [
        f"<svg viewBox='0 0 {width} {height}' width='100%' role='img' "
        f"aria-label='{title}' style='max-width:820px'>",
        f"<text x='{pad_left}' y='24' font-family='system-ui,sans-serif' font-size='15' "
        f"font-weight='600' fill='#134e4a'>{title}（单位：{unit}）</text>",
        f"<line x1='{pad_left}' y1='{pad_top + plot_h}' x2='{width - 20}' y2='{pad_top + plot_h}' "
        "stroke='#99f6e4' stroke-width='1'/>",
    ]

    for index, (label, value) in enumerate(zip(labels, values)):
        bar_h = max(2.0, value / vmax * plot_h)
        x = pad_left + slot * index + (slot - bar_w) / 2
        y = pad_top + plot_h - bar_h
        fill = highlight_color if highlight is not None and index == highlight else color
        parts.append(
            f"<rect x='{x:.1f}' y='{y:.1f}' width='{bar_w:.1f}' height='{bar_h:.1f}' rx='6' fill='{fill}'/>"
        )
        parts.append(
            f"<text x='{x + bar_w / 2:.1f}' y='{y - 6:.1f}' text-anchor='middle' "
            f"font-family='ui-monospace,monospace' font-size='12' fill='#475569'>{value}</text>"
        )
        parts.append(
            f"<text x='{x + bar_w / 2:.1f}' y='{pad_top + plot_h + 20}' text-anchor='middle' "
            f"font-family='ui-monospace,monospace' font-size='12' fill='#475569'>{label}</text>"
        )

    parts.append("</svg>")
    return "".join(parts)


recent = days[-7:]
labels = [date.fromisoformat(day).strftime("%m-%d") for day in recent]
values = [by_day[day] for day in recent]
display(HTML(bar_chart(labels, values, title="最近 7 天专注时长", color="#0d9488", highlight=len(labels) - 1, unit="分钟")))

subject_rows = sorted(by_subject.items(), key=lambda item: -item[1])
display(
    HTML(
        bar_chart(
            [name for name, _ in subject_rows],
            [minutes for _, minutes in subject_rows],
            title="各科目投入时长",
            color="#2dd4bf",
            unit="分钟",
        )
    )
)

## 4. 可选：用 pandas 和 matplotlib

装了这两个库的话，这一节会给出更标准的表格和图；没装就会提示安装命令，不影响前面的结果。

In [6]:
try:
    import matplotlib
    import pandas as pd

    matplotlib.rcParams["font.family"] = "sans-serif"
    matplotlib.rcParams["axes.unicode_minus"] = False

    df = pd.DataFrame(records)
    df["day"] = pd.to_datetime(df["day"])
    summary = (
        df.groupby("subject")
        .agg(总时长=("minutes", "sum"), 平均时长=("minutes", "mean"), 完成任务=("tasks", "sum"))
        .sort_values("总时长", ascending=False)
        .round(1)
    )
    display(summary)

    daily = df.groupby(df["day"].dt.strftime("%m-%d"))["minutes"].sum()
    axes = daily.plot(kind="bar", color="#0d9488", figsize=(9, 3.4), rot=0)
    axes.set_ylabel("分钟")
    axes.set_title("每日专注时长")
    axes.spines[["top", "right"]].set_visible(False)
    import matplotlib.pyplot as plt

    plt.tight_layout()
    plt.show()
except ImportError:
    print("尚未安装 pandas / matplotlib，这一节已跳过。")
    print("需要的话在 notebook 里执行：%pip install pandas matplotlib")

尚未安装 pandas / matplotlib，这一节已跳过。
需要的话在 notebook 里执行：%pip install pandas matplotlib


## 5. 换成你自己的真实数据

看板网页把数据存在浏览器的 `localStorage` 里，键名是 `comp5241-study-hub:v1`。导出步骤：

1. 打开看板页面，按 `F12` 打开开发者工具，切到 Console。
2. 粘贴并回车：`copy(localStorage.getItem("comp5241-study-hub:v1"))`
3. 新建一个文本文件粘贴内容，保存为 `study_data.json`，放在本 notebook 同一个文件夹。
4. 重新运行第 1 节，数据来源会自动变成 `study_data.json`。

也可以直接用 CSV（表头 `day,subject,minutes,tasks`），命名为 `study_log.csv` 即可，优先级高于 JSON。

在线模仿一份 CSV：

In [7]:
csv_lines = ["day,subject,minutes,tasks"]
for row in records:
    csv_lines.append(f"{row['day']},{row['subject']},{row['minutes']},{row['tasks']}")

with open("study_log.example.csv", "w", encoding="utf-8") as handle:
    handle.write("\n".join(csv_lines) + "\n")

print("已写出 study_log.example.csv，前 6 行：")
print("\n".join(csv_lines[:6]))

已写出 study_log.example.csv，前 6 行：
day,subject,minutes,tasks
2026-08-31,讲座,60,0
2026-09-01,讲座,90,1
2026-09-01,复习,45,1
2026-09-02,作业,120,2
2026-09-03,讲座,60,0


## 练习

1. **找规律**：哪一天的专注时长明显高于平均？在下面写几行代码，把超过「均值 + 1 个标准差」的日子挑出来。
2. **改口径**：现在的"连续专注"按有记录的天数算。改成"每天至少 60 分钟才算数"，连续天数会变成多少？
3. **看科目结构**：作业和复习的合计占比是多少？如果想把讲座时间压缩一半、把省下的时间给项目，各科占比会怎么变？
4. **换个指标**：用 `tasks` 字段做一张"每天完成任务数"的图，和专注时长图对比，看看"坐得久"和"做得完"是不是一回事。

提示：`statistics.mean(...)`、`statistics.pstdev(...)` 都在标准库里，不需要额外安装。

---

数据来源优先顺序：`study_log.csv` → `study_data.json` → 内置示例。
页面本身在 `web/` 目录，样式和配色与这个 notebook 的图表用同一套 token。